# 04 — Modelling

Trains and evaluates three regression models to predict `pts_next_gw`.

**Validation:** TimeSeriesSplit (5 folds) — never shuffle, always train on past, test on future.

**Models:** Naive baseline → Linear Regression → Random Forest → XGBoost

**Output:** `models/xgboost_model.json`, `models/shap_explainer.pkl`

## 1. Imports, load data, train/test split

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap

PROCESSED = Path('../data/processed')
MODELS    = Path('../models')
MODELS.mkdir(exist_ok=True)

features = pd.read_parquet(PROCESSED / 'features.parquet')
print('Loaded:', features.shape)

FEATURE_COLS = [
    'rolling_pts_3gw', 'rolling_pts_5gw',
    'rolling_minutes_3gw', 'rolling_minutes_5gw',
    'minutes_consistency', 'blank_gw_flag', 'form_streak',
    'rolling_xg_3gw', 'rolling_xg_5gw',
    'rolling_xa_3gw', 'rolling_xa_5gw',
    'xg_overperformance', 'shots_per_90', 'key_passes_per_90',
    'fdr_next', 'fdr_next3', 'is_home_next',
    'opp_goals_conceded_avg', 'has_fixture',
    'value', 'price_change_3gw', 'pts_per_million',
    'ownership_pct', 'ownership_change_3gw', 'is_differential',
    'gw_number', 'games_played',
    'is_gkp', 'is_def', 'is_mid', 'is_fwd',
]
TARGET = 'pts_next_gw'

# Held-out test set: GW 27-30 (last 4 gameweeks)
TRAIN_GWS = features['round'] <= 26
TEST_GWS  = features['round'] >= 27

train_df = features[TRAIN_GWS].copy()
test_df  = features[TEST_GWS].copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

print(f'Train: {X_train.shape}  (GW 2-26)')
print(f'Test:  {X_test.shape}   (GW 27-30)')

## 2. Naive baseline

Predict next GW points = rolling 3-GW average. This is what FPL's own "Form" metric approximates.

In [ ]:
baseline_preds = X_test['rolling_pts_3gw']

baseline_mae  = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = mean_squared_error(y_test, baseline_preds) ** 0.5
baseline_r2   = r2_score(y_test, baseline_preds)

print('=== Naive Baseline (rolling_pts_3gw) ===')
print(f'  MAE:  {baseline_mae:.4f}')
print(f'  RMSE: {baseline_rmse:.4f}')
print(f'  R²:   {baseline_r2:.4f}')
print()
print('Target: beat this MAE with ML models.')

## 3. TimeSeriesSplit — cross-validation setup

Split by gameweek, not by row. We sort rounds first, then use unique GW boundaries
to ensure all players in a GW are always in the same fold.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def gw_time_series_cv(df, n_splits=5):
    """
    Yield (train_idx, val_idx) splits based on gameweek order.
    All players in the same GW always end up in the same fold.
    """
    rounds = sorted(df['round'].unique())
    tss = TimeSeriesSplit(n_splits=n_splits)
    for train_gws, val_gws in tss.split(rounds):
        train_rounds = [rounds[i] for i in train_gws]
        val_rounds   = [rounds[i] for i in val_gws]
        train_idx = df[df['round'].isin(train_rounds)].index
        val_idx   = df[df['round'].isin(val_rounds)].index
        yield train_idx, val_idx

def cross_validate_model(model, df, n_splits=5):
    maes, rmses, r2s = [], [], []
    for fold, (train_idx, val_idx) in enumerate(gw_time_series_cv(df, n_splits)):
        X_tr = df.loc[train_idx, FEATURE_COLS]
        y_tr = df.loc[train_idx, TARGET]
        X_vl = df.loc[val_idx, FEATURE_COLS]
        y_vl = df.loc[val_idx, TARGET]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_vl)

        maes.append(mean_absolute_error(y_vl, preds))
        rmses.append(mean_squared_error(y_vl, preds) ** 0.5)
        r2s.append(r2_score(y_vl, preds))

    return {
        'mae_mean':  np.mean(maes).round(4),
        'mae_std':   np.std(maes).round(4),
        'rmse_mean': np.mean(rmses).round(4),
        'r2_mean':   np.mean(r2s).round(4),
    }

print('CV setup ready. Splits preview:')
for i, (tr, vl) in enumerate(gw_time_series_cv(train_df)):
    tr_gws = sorted(train_df.loc[tr, 'round'].unique())
    vl_gws = sorted(train_df.loc[vl, 'round'].unique())
    print(f'  Fold {i+1}: train GW {tr_gws[0]}-{tr_gws[-1]} | val GW {vl_gws[0]}-{vl_gws[-1]}')

## 4. Linear Regression

In [ ]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])

lr_cv = cross_validate_model(lr, train_df)
print('=== Linear Regression (CV) ===')
for k, v in lr_cv.items():
    print(f'  {k}: {v}')

# Final test score
lr.fit(X_train, y_train)
lr_test_preds = lr.predict(X_test)
lr_test_mae   = mean_absolute_error(y_test, lr_test_preds)
lr_test_rmse  = mean_squared_error(y_test, lr_test_preds) ** 0.5
lr_test_r2    = r2_score(y_test, lr_test_preds)
print(f'\n  Test MAE:  {lr_test_mae:.4f}')
print(f'  Test RMSE: {lr_test_rmse:.4f}')
print(f'  Test R²:   {lr_test_r2:.4f}')

## 5. Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
)

rf_cv = cross_validate_model(rf, train_df)
print('=== Random Forest (CV) ===')
for k, v in rf_cv.items():
    print(f'  {k}: {v}')

rf.fit(X_train, y_train)
rf_test_preds = rf.predict(X_test)
rf_test_mae   = mean_absolute_error(y_test, rf_test_preds)
rf_test_rmse  = mean_squared_error(y_test, rf_test_preds) ** 0.5
rf_test_r2    = r2_score(y_test, rf_test_preds)
print(f'\n  Test MAE:  {rf_test_mae:.4f}')
print(f'  Test RMSE: {rf_test_rmse:.4f}')
print(f'  Test R²:   {rf_test_r2:.4f}')

## 6. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

xgb_cv = cross_validate_model(xgb_model, train_df)
print('=== XGBoost (CV) ===')
for k, v in xgb_cv.items():
    print(f'  {k}: {v}')

xgb_model.fit(X_train, y_train)
xgb_test_preds = xgb_model.predict(X_test)
xgb_test_mae   = mean_absolute_error(y_test, xgb_test_preds)
xgb_test_rmse  = mean_squared_error(y_test, xgb_test_preds) ** 0.5
xgb_test_r2    = r2_score(y_test, xgb_test_preds)
print(f'\n  Test MAE:  {xgb_test_mae:.4f}')
print(f'  Test RMSE: {xgb_test_rmse:.4f}')
print(f'  Test R²:   {xgb_test_r2:.4f}')

## 7. Results comparison

In [ ]:
results = pd.DataFrame([
    {'Model': 'Naive Baseline',    'CV MAE': '—',                       'CV MAE std': '—',            'Test MAE': round(baseline_mae, 4),   'Test RMSE': round(baseline_rmse, 4), 'Test R²': round(baseline_r2, 4)},
    {'Model': 'Linear Regression', 'CV MAE': lr_cv['mae_mean'],          'CV MAE std': lr_cv['mae_std'],  'Test MAE': round(lr_test_mae, 4),    'Test RMSE': round(lr_test_rmse, 4),  'Test R²': round(lr_test_r2, 4)},
    {'Model': 'Random Forest',     'CV MAE': rf_cv['mae_mean'],          'CV MAE std': rf_cv['mae_std'],  'Test MAE': round(rf_test_mae, 4),    'Test RMSE': round(rf_test_rmse, 4),  'Test R²': round(rf_test_r2, 4)},
    {'Model': 'XGBoost',           'CV MAE': xgb_cv['mae_mean'],         'CV MAE std': xgb_cv['mae_std'], 'Test MAE': round(xgb_test_mae, 4),   'Test RMSE': round(xgb_test_rmse, 4), 'Test R²': round(xgb_test_r2, 4)},
])

print(results.to_string(index=False))
print()
best_model_name = results.loc[results['Test MAE'].astype(float).idxmin(), 'Model']
print(f'Best model by test MAE: {best_model_name}')

In [ ]:
# Visualise test MAE comparison
fig, ax = plt.subplots(figsize=(8, 4))
models  = ['Naive Baseline', 'Linear Regression', 'Random Forest', 'XGBoost']
maes    = [baseline_mae, lr_test_mae, rf_test_mae, xgb_test_mae]
colors  = ['#888888', '#4c8fbe', '#5cb85c', '#f0ad4e']

bars = ax.bar(models, maes, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(baseline_mae, color='red', linestyle='--', linewidth=1, label='Baseline MAE')
for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{mae:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('MAE (FPL points)')
ax.set_title('Model comparison — test set MAE (GW 27–30)')
ax.legend()
plt.tight_layout()
plt.savefig(MODELS / 'model_comparison.png', dpi=150)
plt.show()

## 8. XGBoost built-in feature importance

In [ ]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURE_COLS,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 9))
importance.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost feature importance')
plt.tight_layout()
plt.savefig(MODELS / 'feature_importance.png', dpi=150)
plt.show()

print('Top 10 features:')
print(importance.head(10).round(4).to_string())

## 9. SHAP — beeswarm plot

SHAP explains **how much each feature pushes each prediction** above or below
the baseline. The beeswarm shows all players across all test GWs.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test)

plt.figure(figsize=(9, 8))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('SHAP beeswarm — test set (GW 27–30)')
plt.tight_layout()
plt.savefig(MODELS / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. SHAP — waterfall plot for a single player

Pick a recognisable player from the test set and explain exactly why
the model predicted their points that way.

In [ ]:
# Find a high-scoring player in the test set for a clear waterfall
test_with_meta = test_df.copy()
test_with_meta['predicted'] = xgb_test_preds
test_with_meta['shap_idx'] = range(len(test_with_meta))

# Pick the player with the highest predicted score in GW 30
top_player = test_with_meta[test_with_meta['round'] == 30].nlargest(1, 'predicted').iloc[0]
print(f"Player: {top_player['web_name']} | GW: {top_player['round']} | "
      f"Predicted: {top_player['predicted']:.2f} | Actual: {top_player['pts_next_gw']:.0f}")

idx = int(top_player['shap_idx'])
plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_values[idx], max_display=15, show=False)
plt.title(f"SHAP waterfall — {top_player['web_name']} GW{int(top_player['round'])}")
plt.tight_layout()
plt.savefig(MODELS / 'shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save model and explainer

In [ ]:
import pickle

# Retrain on ALL available data (train + test) before saving
X_all = features[FEATURE_COLS]
y_all = features[TARGET]
xgb_model.fit(X_all, y_all)

# Save XGBoost model
xgb_model.save_model(MODELS / 'xgboost_model.json')

# Save SHAP explainer
final_explainer = shap.TreeExplainer(xgb_model)
with open(MODELS / 'shap_explainer.pkl', 'wb') as f:
    pickle.dump(final_explainer, f)

# Save feature column list
import json
with open(MODELS / 'feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('Saved:')
for p in sorted(MODELS.iterdir()):
    print(f'  {p.name:<40} {p.stat().st_size / 1024:.1f} KB')